# Configuration

In [ ]:
!uv pip install datasets transformers accelerate

Using Python 3.11.12 environment at: /usr
Resolved 57 packages in 624ms
⠙ Preparing packages... (0/15)
⠙ Preparing packages... (0/15)
⠙ Preparing packages... (0/15)
⠙ Preparing packages... (0/15)
⠙ Preparing packages... (0/15)
⠙ Preparing packages... (0/15)
⠙ Preparing packages... (0/15)
dill       ------------------------------ 78.93 KiB/113.53 KiB
⠙ Preparing packages... (0/15)
dill       ------------------------------ 78.93 KiB/113.53 KiB
xxhash     ------------------------------     0 B/190.26 KiB
⠙ Preparing packages... (0/15)
dill       ------------------------------ 78.93 KiB/113.53 KiB
xxhash     ------------------------------     0 B/190.26 KiB
nvidia-cuda-cupti-cu12 ------------------------------     0 B/13.17 MiB
⠙ Preparing packages... (0/15)
dill       ------------------------------ 78.93 KiB/113.53 KiB
xxhash     ------------------------------     0 B/190.26 KiB
nvidia-cuda-cupti-cu12 ------------------------------     0 B/13.17 MiB
nvidia-nvjitlink-cu12 -----------------

In [ ]:
import os
import enum
import random
import json

import torch
import transformers
import pandas as pd
from datasets import load_dataset
from google.colab import drive


DATASET = "elife"
SPLIT = "validation"
DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
CHECKPOINT_PATH = os.path.join(DRIVE_ROOT, f"{DATASET}_{SPLIT}_ckpt.parquet")
OUTPUT_PATH = os.path.join(DRIVE_ROOT, f"{DATASET}_{SPLIT}_llama.json")

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


# Datasets

In [ ]:
class Datasets(enum.Enum):
    elife = "BioLaySumm/BioLaySumm2025-eLife"
    plos = "BioLaySumm/BioLaySumm2025-PLOS"


dataset = load_dataset(Datasets[DATASET].value)

README.md:   0%|          | 0.00/689 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/126M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/6.92M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/3.27M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4346 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/241 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/142 [00:00<?, ? examples/s]

# Inference

In [ ]:
SYSTEM_PROMPT = "You are a specialist medical communicator responsible for translating biomedical articles into a clear, accurate 10-20 sentence summary for non-experts. The summary should be at a Flesch–Kincaid grade level of 10–14 and explain any technical terms."
SYSTEM_PREFIX = "Lay Summary:"

In [ ]:
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

llama = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
def generate_summaries(dataset, checkpoint_rate=20):
    if os.path.exists(CHECKPOINT_PATH):
        processed_df = pd.read_parquet(CHECKPOINT_PATH)
        results = processed_df.to_dict(orient="records")
        start_index = int(processed_df["index"].max()) + 1
        print(f"Resuming from index {start_index} (loaded {len(results)} records).")
    else:
        results = []
        start_index = 0
        print("No checkpoint found.")

    for i, row in enumerate(dataset):
        if i < start_index:
            continue

        article = row["article"]
        reference_summary = row["summary"]

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": article},
            {"role": "assistant", "content": SYSTEM_PREFIX},
        ]

        outputs = llama(
            messages,
            max_new_tokens=400,
            continue_final_message=True,
            pad_token_id = llama.tokenizer.eos_token_id,
        )
        generated_summary = outputs[0]["generated_text"][-1]["content"]

        if SYSTEM_PREFIX in generated_summary:
            generated_summary = generated_summary.split(SYSTEM_PREFIX, 1)[-1].strip()
        else:
            generated_summary = generated_summary.strip()

        generated_summary = generated_summary.replace("\n", " ").strip()

        results.append(
            {
                "index": i,
                "document": article,
                "summary": reference_summary,
                "generated_caption": generated_summary,
            }
        )

        if (i + 1) % checkpoint_rate == 0:
            pd.DataFrame(results).to_parquet(CHECKPOINT_PATH, index=False)
            print(f"Checkpoint saved at index {i}.")

    data = [
        {
            "document": r["document"],
            "reference": r["summary"],
            "generated_caption": r["generated_caption"],
        }
        for r in results
    ]

    with open(OUTPUT_PATH, "w") as f:
        json.dump(data, f)

    pd.DataFrame(results).to_parquet(CHECKPOINT_PATH, index=False)

    print("Done! Data saved to:", OUTPUT_PATH)

In [ ]:
generate_summaries(dataset[SPLIT])

Resuming from index 200 (loaded 200 records).
Checkpoint saved at index 219.
Checkpoint saved at index 239.
Done! Data saved to: /content/drive/MyDrive/Colab Notebooks/elife_validation_llama.json
